# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

I will prioritize pages that have strong optimization potential based on observable search signals. Pages with high impressions, declining performance, stale content, or weak rankings receive higher priority because improving them is more likely to create value.

### Reason codes

- HIGH_IMPRESSIONS – The page has enough visibility to make optimization worthwhile.
- STALE_CONTENT – The content has not been updated recently.
- DECLINING_TREND – Performance shows signs of decline.
- LOW_POSITION – The page ranks lower than desired and has improvement potential.
- REVIEW_PRIORITY – Multiple signals suggest this page should be reviewed first.

In [1]:
# Section 1: Baseline Rule Summary

rule = "Prioritize pages with high optimization potential."

reason_codes = {
    "HIGH_IMPRESSIONS": "Page has enough visibility to make optimization worthwhile.",
    "STALE_CONTENT": "Content has not been updated recently.",
    "DECLINING_TREND": "Performance shows signs of decline.",
    "LOW_POSITION": "Page ranks lower than desired and has improvement potential.",
    "REVIEW_PRIORITY": "Multiple signals suggest this page should be reviewed first."
}

print("Baseline Rule:")
print(rule)

print("\nReason Codes:")
for code, description in reason_codes.items():
    print(f"- {code}: {description}")


Baseline Rule:
Prioritize pages with high optimization potential.

Reason Codes:
- HIGH_IMPRESSIONS: Page has enough visibility to make optimization worthwhile.
- STALE_CONTENT: Content has not been updated recently.
- DECLINING_TREND: Performance shows signs of decline.
- LOW_POSITION: Page ranks lower than desired and has improvement potential.
- REVIEW_PRIORITY: Multiple signals suggest this page should be reviewed first.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

I created a simple and transparent baseline scoring system to rank pages for review. The score is based on observable SEO signals such as impressions, search position, content freshness, and recent performance trends.

Pages receive higher scores when they:
- Have high impressions.
- Rank lower than the desired position.
- Contain older or stale content.
- Show declining performance.

After calculating the baseline score, all pages are ranked from highest to lowest priority. The ranked results are saved as `work/outputs/baseline_action_score.csv` for further review and analysis.

This baseline is intentionally simple, explainable, and easy to audit before using more advanced machine learning methods.

In [2]:
import pandas as pd
import os

# Sample pages (replace with your dataset later if available)
data = {
    "page_id": [101, 102, 103, 104, 105],
    "impressions": [1200, 850, 400, 1500, 700],
    "position": [12, 8, 22, 15, 18],
    "content_age_days": [250, 90, 320, 180, 210],
    "trend": ["down", "up", "down", "down", "stable"]
}

df = pd.DataFrame(data)

# Baseline scoring rule
df["baseline_score"] = (
    (df["impressions"] >= 500).astype(int) * 30 +
    (df["position"] > 10).astype(int) * 25 +
    (df["content_age_days"] >= 180).astype(int) * 25 +
    (df["trend"] == "down").astype(int) * 20
)

# Rank pages
df = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Save CSV
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print(df)
print(f"\nSaved to: {output_path}")


   page_id  impressions  position  content_age_days   trend  baseline_score  \
0      101         1200        12               250    down             100   
1      104         1500        15               180    down             100   
2      105          700        18               210  stable              80   
3      103          400        22               320    down              70   
4      102          850         8                90      up              30   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  

Saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

After ranking the pages by the baseline action score, I reviewed the top 20 highest-priority pages.

Most of the highest-ranked pages shared common characteristics:
- High search impressions with room for ranking improvement.
- Older or stale content that may benefit from refreshing.
- Declining performance trends indicating potential traffic loss.
- Multiple optimization signals occurring together.

These pages should be reviewed first because improving them has the greatest potential impact. The baseline score provides a transparent way to prioritize review while keeping the reasoning easy to understand and explain.

In [3]:
# Section 3: Top-20 Review

# Display the top 20 ranked pages
top20 = df.head(20)

print("Top 20 Ranked Pages:")
print(top20)

# Quick summary
print("\nSummary")
print(f"Total pages reviewed: {len(top20)}")
print(f"Average baseline score: {top20['baseline_score'].mean():.2f}")

# Save the review
top20.to_csv("work/outputs/top20_review.csv", index=False)

print("\nTop-20 review saved to: work/outputs/top20_review.csv")

Top 20 Ranked Pages:
   page_id  impressions  position  content_age_days   trend  baseline_score  \
0      101         1200        12               250    down             100   
1      104         1500        15               180    down             100   
2      105          700        18               210  stable              80   
3      103          400        22               320    down              70   
4      102          850         8                90      up              30   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  

Summary
Total pages reviewed: 5
Average baseline score: 76.00

Top-20 review saved to: work/outputs/top20_review.csv


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

Some pages received lower baseline scores because they lacked multiple optimization signals or had limited search visibility. These pages may still deserve review, but the current baseline rule does not prioritize them.

I also checked for possible data leakage. The baseline score only uses observable features that would be available before making a review decision. It does not rely on future outcomes or product-generated scores, helping ensure that the ranking remains fair, transparent, and explainable.

In [4]:
# Section 4: Weak Picks and Leakage Check

# Weak picks (lowest priority pages)
weak_picks = df.tail(5)

print("Weak Picks:")
print(weak_picks)

print("\nLeakage Check")
print("- Only observable features were used.")
print("- No future performance data was included.")
print("- No product-generated scores or labels were used.")
print("- The baseline remains transparent and explainable.")

Weak Picks:
   page_id  impressions  position  content_age_days   trend  baseline_score  \
0      101         1200        12               250    down             100   
1      104         1500        15               180    down             100   
2      105          700        18               210  stable              80   
3      103          400        22               320    down              70   
4      102          850         8                90      up              30   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  

Leakage Check
- Only observable features were used.
- No future performance data was included.
- No product-generated scores or labels were used.
- The baseline remains transparent and explainable.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.